# Retrieval-augmented generation

Suppose a student asks, *How does AI affect inequality?* You would probably want to know which papers the student has read before you explain the mechanisms. A retrieval-augmented generation (RAG) system searches a reading list for passages that bear on a question, then gives the question and those passages to a language model to write its reply. The reading list reaches the model through each request. Adding a paper changes the evidence available on the next search without training the model again.

**Follow one question through the system**

```text
Student's question ──► Search the indexed reading list
        │                            │
        │                    Selected passages
        ▼                            ▼
    Send the original question and passages to the language model
                                │
                                ▼
                 Answer with passage references
```

**Run the example.** Open this file in Jupyter or VS Code, select a Python 3 kernel and choose **Run All**. The notebook builds a small library from three public papers, searches it twice and constructs the first prompt. For an offline run, the embeddings and the model’s original reply are saved in this file. The second turn illustrates what happens when the passages cannot support a requested estimate. To try other questions and generate new replies, use the optional model setting in step 1.

## 1. Start with a reading list

The student’s question could refer to differences among households, among countries or between labor and capital. Our library covers all three. It contains short teaching notes on Korinek and Stiglitz’s work on income distribution, their paper on economic development, and Trammell and Korinek’s work on growth under transformative AI. Each note links to the original paper below.

If you already have a vector database, you can start at the search stage. This example builds its own miniature index so it runs on its own. It splits the notes into six passages, keeps the paper and text location for each, and assigns identifiers such as `distribution-1`. You will see the same identifiers beside the model’s answer.

In [ ]:
papers = [
    {"id": "distribution", "title": "Artificial Intelligence and Its Implications for Income Distribution and Unemployment", "url": "https://www.nber.org/papers/w24174",
     "text": "Two channels link AI to income inequality. Innovators can capture some of the surplus. Changes in wages and returns to capital can also shift income among households with different endowments. Korinek and Stiglitz examine how these channels interact. The size of each effect depends on the economic setting. The argument identifies possible channels of income redistribution."},
    {"id": "development", "title": "Artificial Intelligence, Globalization, and Strategies for Economic Development", "url": "https://www.nber.org/papers/w28453",
     "text": "Korinek and Stiglitz examine how the spread of artificial intelligence could affect developing economies. If automation reduces the value of low labor costs, an export strategy built on inexpensive labor may become less attractive. Countries differ in their production patterns and in their capacity to capture gains from new technology."},
    {"id": "growth", "title": "Economic Growth under Transformative AI", "url": "https://www.nber.org/papers/w31815",
     "text": "Trammell and Korinek study growth when AI automates tasks in production or in research. In their modeling framework, sufficiently broad automation of production can accelerate economic growth and reduce labor’s share of income. A change in the aggregate labor share is conceptually distinct from a measure of inequality across individual households."},
]
question = "How does AI affect inequality?"

In [ ]:
def chunk_note(text, width=768, overlap=50):
    if not 0 <= overlap < width:
        raise ValueError("Overlap must be smaller than the window.")
    start = 0
    while start < len(text):
        end = min(start + width, len(text))
        if end < len(text):
            tail_start = max(start + overlap + 1, end - 100)
            stops = [text.rfind(mark, tail_start, end) for mark in ".!?"]
            if max(stops) >= 0:
                end = max(stops) + 1
        yield start, end, text[start:end]
        if end == len(text):
            break
        start = end - overlap

# A shorter window makes overlap visible in these short teaching notes.
passages = []
for paper in papers:
    for index, (start, end, text) in enumerate(chunk_note(paper["text"], width=245, overlap=50)):
        passages.append({"id": f"{paper['id']}-{index+1}", "title": paper["title"],
                         "url": paper["url"], "start": start, "end": end, "text": text})
for p in passages:
    print(p["id"], (p["start"], p["end"]), p["text"][:60])

### How a passage becomes searchable

Each passage has a text embedding, a list of numbers representing its meaning. MiniLM produced 384 coordinates per passage. The next cells load those numbers and store them beside the original text in a SQLite database in memory. The first question and the follow-up search also have saved embeddings.

Leave `fresh_models = False` for the two worked turns. To ask new questions or change the reading list, set it to `True` and run all cells again. That mode uses PyTorch and transformers 4.57.6, downloads any missing public MiniLM and Qwen model weights, and runs both models locally.

In [ ]:
import hashlib
import json
import math
import sqlite3
import struct

fresh_models = False

In [ ]:
ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
ENCODER_REV = "c9745ed1d9f207416be6d2e6f8de32d1f16199bf"
GENERATOR = "Qwen/Qwen2.5-0.5B-Instruct"
GENERATOR_REV = "7ae557604adf67be50417f59c2c2f167def9a775"


def encode_texts(texts):
    import torch
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(ENCODER, revision=ENCODER_REV, token=False)
    model = AutoModel.from_pretrained(ENCODER, revision=ENCODER_REV, token=False).eval()
    batch = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.inference_mode():
        hidden = model(**batch).last_hidden_state
        mask = batch.attention_mask.unsqueeze(-1)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        return torch.nn.functional.normalize(pooled, dim=1).cpu().numpy()


def load_writer(local_model=None):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    name = local_model or GENERATOR
    options = {"token": False, "revision": GENERATOR_REV} if local_model is None else {"local_files_only": True}
    tokenizer = AutoTokenizer.from_pretrained(name, **options)
    model = AutoModelForCausalLM.from_pretrained(name, **options).eval()

    def write(prompt, max_new_tokens=165):
        chat = [{"role": "system", "content": "You are a careful economics research assistant."},
                {"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        tokens = tokenizer(formatted, return_tensors="pt")
        with torch.inference_mode():
            result = model.generate(**tokens, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(result[0, tokens.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return write

In [ ]:
saved_input_hash = '743f7b2dd651811361ef003ae87be059a21365446e5fd6747bec98e9bdd73645'
saved_prompt_hash = '0bb177e6f306c103aab893605618d27cb3fe0ec2a1fceb695b58f258d42f0b9e'
saved_answer = "[development-1] Korinek and Stiglitz examine how the spread of artificial intelligence could affect developing economies. If automation reduces the value of low labor costs, an export strategy built on inexpensive labor may become less attractive. [distribution-1] Two channels link AI to income inequality. Innovators can capture some of the surplus. Changes in wages and returns to capital can also shift income among households with different endowments. [growth-2] economic growth and reduce labor's share of income. A change in the aggregate labor share is conceptually distinct from a measure of inequality across individual households."
saved_vectors = [[-0.02999974, -0.04512062, 0.02645796, -0.03005211, 0.02143897, 0.05777871, -0.03576547, -0.0119017, -0.01190059, 0.01564427, -0.01715533, -0.01680988, -0.00602963, -0.02232129, 0.00377322, 0.00560903, -0.00764736, 0.0514177, -0.0532708, -0.03145615, 0.03347958, -0.04590305, -0.04234902, -0.04344616, 0.03140684, -0.03171561, 0.02432667, 0.02075007, 0.03344128, -0.01610268, 0.03520773, 0.00795811, 0.0935144, -0.0051665, -0.0821631, 0.02593763, -0.00577495, 0.02095802, 0.03717196, -0.01285213, 0.02543216, -0.06468522, -0.01202915, -0.04087934, 0.00847885, 0.01110603, 0.02238736, 0.00197158, -0.03681978, 0.00124976, -0.08730355, 0.04688651, -0.01895767, 0.08653449, -0.06712384, 0.04677846, 0.01057944, 0.02630155, -0.01683187, 0.01759538, 0.06906506, -0.00908169, 0.07564017, 2.114e-05, 0.06320177, -0.03557427, -0.05762734, 0.09256689, -0.12705131, 0.0243758, 0.04780185, -0.07202066, -0.1074877, -0.06255952, 0.1018945, -0.06107475, 0.05303704, 0.0363572, 0.03509279, -0.03033268, -0.00955936, -0.0098876, -0.10463531, 0.04819778, -0.0446241, 0.02824107, 0.01331261, -0.08268389, 0.09660789, -0.00099467, -0.00253309, -0.0265541, -0.01398372, 0.03271852, 0.04143684, -0.03926361, -0.06317203, -0.02444166, 0.01487307, 0.07692695, -0.01086808, -0.03753537, 0.01787688, -0.024768, -0.03901358, -0.0704809, 0.0658544, -0.00449697, 0.00298832, -0.00158551, 0.02397846, 0.05816839, 0.06443661, 0.02148485, 0.06189805, -0.01782597, 0.04372763, 0.07374241, 0.01002173, 0.03137959, 0.03198371, -0.06703406, -0.02457854, 0.06830727, -0.0267532, 0.01701265, -0.07220056, 0.0, -0.01115117, -0.02021763, 0.03052223, 0.0146561, -0.00634384, -0.04768577, -0.04451852, -0.03310858, 0.01870643, -0.00299438, -0.0418081, 0.1054095, -0.05796463, 0.07766099, 0.10149357, -0.00640129, -0.14885677, 0.04370367, 0.0625345, 0.00372871, -0.00013798, -0.12713125, 0.0282877, 0.01206554, 0.05567343, -0.02221846, -0.0280922, -0.025248, 0.0272983, -0.01534012, 0.03043796, 0.06877471, -0.00790481, -0.06515438, -0.0844595, 0.02205383, 5.606e-05, 0.00633044, 0.00417685, 0.03364765, -0.0490204, -0.00992118, 0.06032808, -0.03937701, 0.04005705, -0.05035024, 0.09180701, 0.03431309, -0.00445953, 0.0475872, -0.02259003, 0.0295488, -0.0225963, -0.10050555, 0.02550319, -0.030387, -0.06242806, 0.06988089, -0.00763909, -0.06736083, -0.07244121, -0.04577444, 0.03051476, 0.04867671, -0.01136813, 0.16062161, 0.02815342, 0.05386198, 0.03096911, 0.08545116, -0.05045498, -0.01468576, -0.13134608, 0.05860084, -0.06608618, 0.05120844, -0.11078991, -0.04972881, 0.0900994, -0.00435535, -0.05219285, 0.0105312, 0.02756104, -0.03355274, 0.05296231, 0.02449558, 0.05721474, -0.01425737, 0.1346824, -0.03029713, 0.01720425, -0.00671548, 0.01281499, 0.07077017, 0.04103493, -0.0, -0.07541265, 0.00913492, -0.11890747, -0.01276756, 0.04088069, -0.00991706, 0.03522961, -0.10305184, -0.03185224, 0.04245961, -0.06483498, -0.01940493, -0.03342409, 0.07250854, -0.01309939, -0.11428316, 0.06134424, -0.02151217, 0.03922294, -0.06269684, 0.09351566, 0.06543443, 0.02481045, 0.02220932, -0.01136588, 0.00932011, -0.11766089, 0.1359095, -0.05388027, -0.02653771, 0.02595571, -0.01512235, -0.02984311, 0.00096793, 0.02916562, 0.03351879, 0.00546297, -0.05491066, -0.04755796, 0.00632315, -0.0219801, -0.08047119, 0.00824687, -0.00231306, -0.05169436, 0.02412433, 0.02039506, -0.00440954, 0.02346239, -0.05992948, 0.11763156, 0.0421301, 0.03619757, 0.00508004, -0.09529188, 0.03912463, 0.06429491, -0.00774546, 0.02593297, 0.02985103, -0.10041504, -0.018511, -0.05356941, 0.04784394, -0.03322638, -0.04699259, 0.05406325, -0.05779728, 0.01689549, -0.07547351, 0.08511039, -0.07644115, 0.0612371, -0.05974614, -0.02431422, 0.11184833, -0.01427265, 0.05676137, 0.0098714, -0.03839193, -0.05169797, -0.07171486, 0.09585895, -0.02850332, -0.05105552, 0.03797064, -0.06163814, -0.04305866, -0.02424369, -0.01563673, -0.07404698, 0.01186411, -0.05094064, -0.05156478, 0.01139822, -3e-08, 0.0307118, -0.0251522, 0.0195733, 0.00349315, -0.04652858, -0.09256136, -0.05319797, 0.02218799, 0.00504129, 0.01789565, -0.01503985, 0.0652303, 0.0796098, 0.04271462, 0.07628267, -0.01377056, -0.04890946, 0.00942522, -0.03228326, -0.04073916, 0.12043999, -0.01547383, 0.03179424, -0.05626207, -0.01700216, -0.00624797, -0.0052622, 0.0648802, -0.04165706, 0.08262432, 0.01640183, 0.03094358, 0.06948344, -0.04498351, 0.06109053, 0.05859551, 0.01942085, -0.03218304, -0.00299291, -0.03775007, 0.0331217, -0.025044, -0.02649511, -0.04808367, 0.02386768, -0.02821701, -0.02387605, -0.01450628, 0.057486, -0.07396742, 0.03019904, 0.00411027, 0.03880902, -0.02723117, 0.04628131, -0.12213404, 0.01408502, -0.020483, -0.04268673, 0.02788856, 0.05153823, 0.01612826, 0.03515647, -0.09050034], [-0.01009145, -0.01377692, -0.0567002, -0.03102965, -0.00768567, 0.02825523, 0.00350707, 0.01677621, -0.04985428, -0.03693245, 0.01760557, 0.01498245, -0.01768461, -0.01296923, 0.00211962, -0.03592779, 0.01116928, 0.02603552, -0.02271101, 0.01018617, 0.05377694, -0.09696194, -0.05522527, -0.02060867, 0.09217475, -0.08884175, 0.03238774, -0.02074736, 0.03204852, 0.00876577, 0.06119378, 0.00750678, 0.05659512, 0.03377579, -0.03418809, -0.01662552, 0.0174429, 0.00488329, -0.03818954, 0.04642115, 0.01925095, -0.01169206, -0.03085794, 0.01237938, -0.00226054, -0.02873974, 0.1010384, 0.04758488, -0.08153515, 0.03936682, -0.03348945, 0.04010669, -0.01016896, 0.08743582, 0.00029037, 0.08308975, -0.02236137, -0.01016477, 0.00187442, 0.00201916, -0.0285012, 0.03198735, -0.00492738, 0.03114403, 0.12325627, 0.01649188, 0.00038766, 0.03927249, -0.09861897, -0.02739106, 0.0204662, -0.06812848, -0.06916249, -0.03767779, -0.00933576, -0.1244669, 0.00574695, 0.05819016, -0.02352243, -0.02257026, -0.03568745, -0.01199015, -0.12275689, -0.02348102, -0.05368387, 0.04083404, -0.03753963, -0.08891673, 0.05922649, 0.02988042, 0.02496102, 0.01566667, -0.00552273, 0.01451473, -0.05694466, -0.03499714, 0.01752651, -0.06160222, -0.01587539, 0.09579234, -0.05901597, -0.06158976, 0.05629003, -0.01580066, -0.07715215, -0.10429936, -0.02762894, 0.04456042, -0.05685402, 0.02539964, -0.03414401, 0.00143293, 0.00589983, 0.0133908, 0.03818865, -0.03186484, 0.06156825, -0.02298931, -0.04773759, -0.01881329, 0.10439493, -0.03431872, -0.00540633, 0.03271847, -0.01104375, 0.00454516, 0.01912018, 0.0, 0.02191094, -0.02895117, 0.03444481, 0.00447705, -0.04940905, -0.02354357, -0.03261361, 0.01422759, 0.05460088, 0.00066708, 0.04825938, 0.09313118, 0.01003622, 0.02216625, 0.06092475, 0.02308612, -0.10821982, 0.06944173, 0.06738453, 0.0312492, 0.05613483, -0.00265499, 0.03695245, 0.021473, 0.00678094, -0.04016682, -0.09365544, 0.00112592, -0.01178889, 0.02647122, 0.07746197, 0.0395261, 0.01693537, -0.12587324, -0.09163127, -0.05460946, -0.03850381, -0.01712057, 0.01935432, -0.03523643, -0.03036724, -0.08400051, 0.11077517, 0.01999397, 0.01440065, 0.02421108, 0.06536756, 0.08632797, 0.0592085, 0.02186595, 0.01378135, -0.01657762, 0.04151625, -0.05030355, -0.0062836, 0.0196418, -0.0477462, -0.0066647, -0.04090098, -0.0524173, -0.07110991, -0.08090976, 0.01928449, 0.01620117, -0.00673622, 0.07840388, -0.01390125, 0.05505057, 0.03689022, 0.02785663, -0.01991591, 0.01242516, -0.06485271, 0.06858831, -0.03572727, 0.02201445, -0.07226828, 0.02195757, 0.01400396, 0.01441801, -0.0434866, 0.00164387, -0.02007327, -0.04041917, 0.06116255, 0.0505272, 0.03478866, 0.01428277, 0.09559266, 0.01208947, 0.05108244, 0.00355765, 0.02260821, -0.06184122, 0.09083367, -0.0, -0.0911812, 0.02474563, -0.08832322, 0.03862508, 0.02926525, 0.02393772, 0.05475602, -0.06111152, -0.02928554, 0.06140342, -0.05919515, -0.0064547, 0.02993031, 0.02226311, -0.04020957, -0.05266022, 0.04688226, -0.01605468, 0.0751581, -0.0523573, 0.07192816, -0.00876656, 0.03717535, 0.03599745, 0.00748234, 0.01218551, -0.08580052, 0.09911568, -0.0568743, 0.0022791, 0.0591937, 0.03163726, -0.02249116, -0.05376096, 0.04866458, 0.00864661, -0.07678252, 0.04847332, -0.06528077, -0.08237389, -0.00442106, -0.0704055, -0.02419415, -0.02486455, 0.01839855, 0.01623148, -0.01308323, -0.02419357, 0.06412463, -0.02660419, 0.08811284, 0.08813673, 0.10123578, 0.02636314, -0.03812203, 0.01316776, 0.0346675, 0.00570394, 0.0747547, -0.0073273, -0.03336278, -0.00889117, -0.05304571, 0.08384959, 0.01159924, -0.1335686, 0.02333921, -0.12472218, 0.05184409, -0.06054976, 0.03905504, -0.16290757, 0.05596427, -0.09528308, -0.02795432, 0.18270221, -0.00104897, 0.10969114, 0.01231987, 0.02459865, -0.04903128, -0.00822866, 0.0685099, -0.03044528, -0.09485978, -0.02489824, -0.00731023, -0.05418726, 0.00676343, 0.02237695, -0.04004706, 0.06431073, 0.06087163, -0.03968408, 0.10392427, -4e-08, 0.09352613, -0.00561367, -0.0049455, -0.04091923, -0.05548992, -0.08924761, 0.00848511, -0.0221034, -0.04617098, 0.02903853, 0.01255831, 0.09066279, 0.11984365, 0.02704335, 0.00917134, -0.03369052, -0.04108411, -0.02886685, -0.07598393, -0.03244592, 0.09161872, -0.04735665, -0.0101091, -0.03060068, -0.03917833, 0.0851527, 0.00299231, 0.04940732, -0.03321446, 0.05887331, 0.03722113, -0.03580417, -0.01018882, -0.01744041, -0.00103718, 0.0627224, -0.02880072, -0.00068315, 0.02340372, 0.01590274, 0.00939864, -0.09598585, -0.01133948, -0.01403029, -0.02346338, -0.01159022, -0.0459509, 0.0345713, 0.02348733, -0.04221129, -0.01244725, 0.00800394, -0.06751927, -0.13125758, -0.00570985, -0.12785897, -0.01517706, 0.0008662, -0.03709428, -0.04196576, 0.02604224, -0.01000963, -0.01548054, -0.04047891], [0.01169767, 0.00897172, 0.00136444, 0.01742081, 0.08989918, 0.06382608, -0.04652132, 0.07210016, -0.07339437, 0.02909966, -0.00130162, 0.02585317, -0.00018635, -0.0312543, -0.0498693, 0.00321651, -0.00243868, -0.07407049, -0.06032552, -0.08239325, -0.01936425, -0.07051829, -0.01293626, -0.05834892, 0.02805175, -0.01649329, 0.01906642, -0.0661308, -0.00910937, 0.03072525, -0.02332629, 0.04250259, 0.07708791, 0.01696282, 0.0265702, 0.01129238, 0.02189063, 0.05094785, 0.08217917, -0.05343264, 0.02136786, -0.07981564, -0.05608711, 0.00744027, 0.0069501, 0.05197086, 0.01402264, -0.01257806, -0.07307332, -0.00077582, -0.10315549, -0.02081466, 0.02774571, -0.05757846, -0.01099817, 0.0049762, -0.01714676, -0.0232407, 0.04084349, 0.04525222, 0.01102683, -0.08715817, 0.07795504, -0.02041469, 0.0950539, -0.01461856, -0.06096576, 0.00666823, -0.10736942, 0.02654627, 0.1183015, 0.00921662, -0.04570818, -0.00539691, -0.01589943, -0.1313135, -0.02207643, -0.00467387, 0.04336214, -0.02212097, -0.02007278, -0.04801326, -0.12900098, 0.07774904, -0.0883045, -0.00226187, -0.02053992, -0.11010549, 0.06938807, -0.00706345, 0.01360628, -0.09906545, -0.03287566, 0.03480212, 0.03409622, 0.02867246, 0.01715898, 0.0405297, -0.02345968, 0.0386929, 0.0149211, -0.03844547, 0.03615823, -0.01568047, -0.0706111, -0.03024866, 0.00012169, -0.07849459, 0.02886955, 0.00121521, -0.0583474, 0.0345155, 0.0227186, -0.01402225, -0.00589658, -0.04682522, 0.01089685, 0.07059821, -0.01634072, 0.03352133, 0.07182085, -0.02573194, -0.06166178, 0.09994463, -0.00174079, -0.02769103, -0.06757197, 0.0, -0.09795441, 0.0088519, -0.01791173, 0.04058728, -0.02478671, -0.04014361, 0.0423065, -0.00554333, 0.02737344, 0.00584419, -0.1075041, 0.03605048, -0.03704436, 0.08588668, 0.07119033, -0.04572092, -0.01604295, -0.00878844, 0.04417204, -0.00642605, 0.06421797, -0.09136032, -0.02877778, 0.00827382, 0.01260815, 0.01138002, -0.01451266, -0.04927008, 0.08988247, 0.01874117, -0.02343709, 0.07344659, -0.01654257, 0.03767907, -0.06279377, 0.00092967, -0.08892864, 0.01138028, -0.01768006, 0.09905835, -0.05501242, 0.0338652, 0.00397551, 0.0164056, 0.06952699, 0.03557514, 0.06720327, -0.02552214, 0.06543993, -0.01117684, -0.07772536, 0.03688842, 0.09991196, -0.10828074, 0.07937966, 0.00323526, -0.03038732, -0.04569406, 0.00448377, -0.03126479, -0.04003668, -0.04125963, 0.02712424, 0.07667101, 0.03291997, 0.06667674, 0.03017904, 0.03842961, 0.04971553, 0.0525065, 0.02049225, 0.06699785, -0.08188546, -0.02106725, -0.00578182, 0.10430972, 0.00826546, -0.01195368, 0.0262714, -0.1130388, -0.06400372, -0.00396522, -0.06155497, -0.05973628, 0.08090072, 0.04565432, -0.01312533, 0.00850628, 0.05727913, 0.06412101, -0.01452746, -0.05970128, -0.07596789, 0.02882341, -0.00743194, -0.0, -0.10855122, -0.00825902, -0.08660536, 0.07388495, -0.03023261, 0.06534071, 0.0285541, -0.01384851, 0.00436262, 0.05515708, -0.01804431, -0.02821819, -0.01657385, 0.03398532, 0.00498933, -0.01738474, 0.0290488, 0.0165624, 0.05333858, -0.00220106, 0.00649093, 0.02192648, -0.02762584, -0.01239242, 0.02851102, 0.08833678, -0.11215574, 0.02448746, -0.02858365, 0.05128226, 0.01685778, -0.00977079, -0.00931093, 0.12209357, -0.01762026, 0.07511923, -0.06525007, 0.02287961, 0.01468072, 0.00808074, -0.00346855, -0.03580624, -0.0466205, -0.00924424, -0.04548457, -0.02491089, -0.05373588, 0.02219211, 0.04749329, -0.03686104, 0.12670057, 0.08798875, -0.03482041, -0.10782882, -0.11887483, -0.00182101, 0.03343076, -0.01356974, -0.04337331, 0.04375382, -0.06150147, 0.00894001, 0.06738694, 0.01666904, -0.02930881, -0.04312311, 0.0320931, 0.06621421, 0.0373765, -0.06234751, 0.02037622, -0.02305429, -0.01360202, 0.02801997, -0.02542338, 0.09743825, -0.0242758, 0.07619091, 0.00293893, 0.06202212, 0.0283434, -0.03535448, 0.10602148, -0.0391598, -0.0430243, 0.02583641, -0.04324197, -0.03877095, -0.0091817, 0.02212588, -0.10785451, 0.00464791, 0.01961325, -0.033253, -0.0681433, -4e-08, 0.02494431, -0.01456797, 0.09715756, 0.02629318, 0.04922902, -0.06905323, -0.07061914, 0.0278737, -0.03280544, 0.04707436, -0.06341852, 0.00149724, 0.08774118, 0.07990153, 0.01844657, 0.08289254, -0.01304139, -0.00419564, -0.02262885, 0.0013383, 0.10873336, 0.04521143, -0.07381967, -0.01852334, 0.00049099, 0.00054379, -0.04036579, 0.02161321, -0.03673884, 0.06791857, -0.03203004, 0.00416433, 0.00992242, 0.01854528, 0.04008345, 0.03938, -0.02137821, -0.08211005, -0.06083133, -0.0343793, -0.00906071, 0.06012866, -0.05730674, -0.02850583, 0.05892638, -0.10541452, -0.0505675, -0.10947423, 0.05657244, 0.01130916, 0.0009184, 0.02679153, 0.01810596, 0.03540391, 0.12171032, -0.0542336, 0.01146446, -0.01511081, -0.08502576, 0.04791852, -0.02857378, 0.00784545, 0.04927018, -0.01586554], [0.01312115, 0.00924953, 0.04250201, 0.03954896, 0.11323447, -0.02289413, -0.06235941, 0.02531094, -0.02786334, 0.01274358, 0.03859294, 0.02561674, 0.01503555, 0.0120904, 0.02828137, -0.07453086, 0.05753335, -0.06835957, 0.02678749, -0.08260147, 0.00089342, -0.07341963, 0.00775226, -0.01954803, 0.03961282, -0.07315724, -0.00353431, -0.02017228, 0.0209465, 0.03808203, -0.12411015, 0.13642913, -0.03491579, 0.0103377, 0.01535554, -0.04780508, -0.01219772, 0.05976558, -0.00036272, -0.02668184, -0.02560043, -0.09842463, -0.01505184, -0.01146998, -0.04123791, 0.0066159, 0.06275515, -0.08396047, -0.11361121, -0.03927889, 0.05747095, -0.00099693, -0.0483593, -0.06619743, 0.00986282, 0.0065684, -0.02016163, 0.0088771, -0.01432794, 0.0148192, 0.00081455, -0.03839689, 0.00449498, -0.01336899, 0.08770778, -0.05327648, -0.03418137, 0.0539144, -0.11073544, -0.00448191, 0.03590154, -0.05132192, -0.1067082, 0.07355235, -0.01012713, -0.11599473, 0.08686126, 0.0381767, -0.0189319, 0.0311446, 0.01731889, -0.01422338, -0.08014402, -0.0251219, -0.03007029, -0.06696573, 0.00125723, -0.09553584, 0.07816952, -0.0298021, 0.02396013, 0.04789501, -0.05155277, 0.05164253, -0.0073752, -0.0444162, 0.03468473, 0.06291973, -0.018791, 0.01142274, 0.00128734, 0.0340137, -0.00059331, 0.04095039, -0.0683092, 0.00803878, -0.05105359, -0.01021433, 0.03736238, 0.03010361, -0.04718399, 0.07434753, -0.04252508, 0.01332406, -0.04394729, -0.08390449, -0.01383693, -0.02901086, 0.00239971, 0.10345441, 0.0388804, -0.05555836, -0.12612005, -0.0002864, 0.01168378, -0.05731902, -0.0243441, -0.0, 0.03761398, 0.03991345, -0.00222865, 0.02659931, 0.00472777, 0.0259236, -0.02542645, 0.00913224, 0.01843976, -0.01610618, -0.13270377, -0.00737461, -0.02373649, 0.04712634, 0.09318926, -0.04565956, -0.02092807, 0.02705417, 0.02221468, 0.05847402, -0.05757316, 0.01521429, -0.00541236, 0.01112412, 0.03482039, -0.04220108, 0.00512554, -0.00198603, -0.02769859, 0.00852373, 0.00524212, 0.04464017, 0.02923645, -0.05806834, -0.07279447, -0.00634657, 0.01633702, 0.0029364, 0.01389224, -0.00434412, -0.07449345, 0.06232929, 0.01781619, -0.02502941, 0.08477562, 0.05166738, 0.1202161, -0.01949799, -0.06210159, 0.04269696, -0.03648155, 0.06572927, 0.03050569, -0.05768606, 0.05952613, -0.02030702, -0.01689124, -0.01460891, 0.02871903, 0.00393229, -0.04194302, -0.0429437, 0.01190385, 0.02382914, 0.04723524, 0.09347752, 0.04991194, 0.01969575, -0.04926315, 0.0722257, 0.00194649, 0.03385461, -0.03857491, -0.02244739, 0.06806005, 0.09139334, 0.06242772, 0.02274135, 0.0685965, -0.0991057, -0.05223657, -0.01284479, 0.01147068, -0.09194596, 0.05242606, -0.0098239, 0.02590575, -0.02658902, 0.10094013, 0.01266894, -0.03387809, -0.01227394, -0.06444908, 0.05461644, 0.09293493, -0.0, -0.0185445, 0.01861501, -0.03443718, 0.05473977, -0.00106124, -0.0240656, 0.03258874, -0.03468508, -0.01221037, 0.05526091, 0.02141388, -0.05498188, 0.00268549, -0.01202719, -0.02103814, -0.04617646, 0.01478909, 0.01136462, 0.00503507, 0.01149116, 0.00689905, 0.05726377, -0.01150156, -0.01835721, -0.02751025, 0.03538121, -0.20169775, 0.03372217, -0.01864464, -0.03039382, -0.05881413, -0.02619181, 0.00462976, 0.06662274, -0.0122433, 0.09238362, -0.04055012, 0.02363357, 0.03754757, 0.04202839, 0.01203608, -0.0432801, 0.07841134, -0.00405119, -0.09431501, -0.04208585, -0.03649964, -0.01841037, 0.06732842, -0.07832978, 0.18134776, 0.05713096, 0.00647106, -0.02409692, -0.04801336, -0.00611364, 0.07961795, -0.0018245, -0.09688905, 0.00561327, -0.01976506, 0.00792595, -0.00547855, 0.05915212, -0.00665729, -0.0413282, 0.06842717, 0.07145578, 0.09316259, -0.04300663, 0.03143627, -0.00395599, -0.00596425, -0.04415775, 0.02129452, 0.00551, 0.04569147, 0.02755349, 0.0122114, 0.00169105, -0.00091192, 0.01504053, 0.07851357, -0.10495782, -0.05724152, 0.08396292, -0.05708906, -0.05681743, -0.04352485, 0.01811374, -0.12704214, -0.00602207, 0.02522578, -0.01880197, -0.01898977, -3e-08, 0.0202918, -0.07002679, 0.02953097, 0.03518579, -0.00779536, -0.10883275, -0.03366332, 0.07377996, 0.07123303, 0.05637209, -0.07730654, -0.02076185, 0.0982521, 0.06628799, 0.07164488, 0.07865299, -0.10134523, -0.01777026, -0.03285272, -0.04560215, 0.04050848, 0.06848121, 0.00882274, -0.01966359, -0.0292847, 0.07694792, -0.05393808, 0.06797997, 0.03612511, 0.04266046, -0.00245554, -0.01857398, -0.02683469, 0.00097655, -0.00051639, -0.0364268, -0.02478441, -0.05232184, -0.0189778, -0.05081791, -0.02677853, 0.01647191, -0.04898483, 0.02711517, 0.02229585, -0.15784855, -0.04558414, 0.01843853, -0.02510949, 0.00280496, 0.01058145, 0.02605448, -0.01650535, -0.07757287, 0.04603704, -0.05244645, 0.04136528, -0.02603201, 0.01966896, 0.03277238, 0.03043594, -0.10350231, 0.06849217, -0.02169083], [-0.05006869, -0.05369638, -0.01115942, 0.04439979, 0.02965586, -0.0102027, -0.04658198, 0.02065326, -0.03655217, 0.02243403, -0.05294289, 0.00634876, -0.03601941, -0.00751048, -0.10695107, 0.05149091, -0.00271376, -0.04954756, -0.04433068, -0.1398782, 0.01890854, -0.10995772, 0.00785427, -0.04484026, -0.02840156, -0.02122043, -0.00084748, -0.04913358, -0.03335726, 0.00142122, 0.00371904, 0.04200182, 0.10009902, 0.01209645, 0.02387918, 0.03343768, 0.00494233, 0.01396351, 0.05366514, -0.025292, 0.05541157, -0.08128037, -0.04601324, -0.03161978, 0.05244762, 0.02042565, -0.01904393, -0.03286906, 0.00167304, 0.03941263, -0.09782203, -0.01637084, 0.06070166, -0.03994592, -0.00140846, 0.06885719, 0.03459132, -0.02395556, 0.01606023, 0.01341894, 0.02797629, -0.0224729, 0.01675074, -0.04915568, 0.08515681, -0.00419358, -0.08421489, 0.03384204, -0.08158107, 0.04526677, -0.01338121, -0.0179781, -0.03990442, 0.04138135, -0.00762954, -0.09712621, -0.01529481, -0.00497834, 0.07327805, -0.04379987, -0.03519021, -0.03901832, -0.12520926, 0.03998177, -0.09549759, 0.0670196, 0.03261404, 0.00800714, 0.12523687, 0.02634647, 0.04804585, -0.04752778, -0.06070321, 0.01631054, -0.02559718, 0.01851243, 0.02826726, -0.06979585, -0.00812723, 0.01901828, 0.00063031, -0.04701274, 0.06577937, -0.03008255, -0.04141491, -0.04998159, 0.00476011, -0.01135717, 0.01515879, -0.01206841, -0.02480526, 0.01113976, -0.01659684, 0.03222435, 0.01648851, -0.01417199, -0.00110342, 0.02144875, 0.04845494, 0.04161542, 0.07027752, -0.01597198, -0.04150628, 0.09226278, -0.0004926, -0.01894847, -0.00685939, 0.0, -0.06186369, -0.02662171, 0.0527027, 0.00492974, -0.02483365, -0.08023917, 0.00532016, 0.02370049, 0.04766873, -0.02657443, -0.10587024, -0.00392346, -0.00793659, 0.03315302, 0.06996998, -0.03811871, 0.01286715, 0.01913517, -0.00376324, -0.01598411, 0.0341832, -0.0517741, -0.00965356, 0.01569397, 0.0415731, 0.01896898, 0.03834871, -0.08212081, 0.05766536, 0.02840628, 0.03279404, 0.06873582, -0.05495729, 0.03644852, -0.14219961, 0.05175417, -0.069328, -0.02928183, 0.00217282, 0.13841322, 0.02293373, -0.01214377, 0.05495445, 0.01512912, -0.06257041, 0.02556232, 0.06487598, 0.01143521, 0.02109093, 0.01005457, -0.01798906, 0.03518817, 0.07649101, -0.16192375, 0.08136827, 0.05026031, -0.01736589, -0.03178789, 0.05873781, -0.01450069, -0.09034384, -0.0677147, -0.00789351, 0.11291541, 0.02544063, 0.12629022, 0.04207123, 0.030796, 0.10198449, 0.05250079, -0.02570218, 0.03594713, -0.03069311, 0.02690483, 0.0258592, 0.01644399, 0.02099695, -0.08231894, -0.0724741, 0.01721014, -0.05437057, 0.01448262, -0.02992432, -0.05206643, 0.09419216, -0.01901082, -0.04199914, 0.09258994, 0.05051016, 0.03523338, -0.01134299, 0.00046189, -0.0747844, 0.09733567, -0.00710765, -0.0, -0.02981953, -0.00065389, -0.07907999, 0.06028974, 0.01990164, 0.00975242, -0.01612879, -0.08432262, -0.0195865, 0.09193466, -0.06274474, 0.00438487, -0.00759952, 0.0505577, -0.02113291, -0.04908288, -0.00282277, -0.00946392, -0.00493942, -0.01259122, -0.03336858, 0.0366227, -0.01332751, 0.01465301, 0.04345353, 0.03293995, -0.11073806, 0.07588772, -0.05685003, 0.02493805, 0.02133059, -0.01824744, 0.01432237, 0.02394868, 0.05042286, 0.04343933, -0.04996513, 0.0020977, -0.03094926, 0.00281725, 0.03280103, -0.06673656, -0.07720128, -0.00887816, -0.04521343, -0.06607317, -0.04060129, 0.02803796, 0.01935421, -0.05043834, 0.0716026, 0.03743441, -0.03443678, -0.12090712, -0.04621978, 0.0127259, 0.01006796, -0.04509874, -0.09291844, 0.02470108, -0.04571864, -0.05947239, 0.08847901, 0.05576969, -0.01550323, 0.00235575, 0.09400696, 0.00188334, 0.00304271, -0.0754668, 0.04721867, 0.00484676, 0.05905698, 0.06515011, -0.05211369, -0.00532796, -0.0166942, 0.02334404, 0.01334161, -0.026485, -0.04776153, -0.03514864, 0.07644152, 0.02935788, -0.13670327, 0.03480398, -0.03781034, 0.03319038, -0.02685575, 0.02897754, -0.03184233, 0.00607014, -0.03650571, 0.04523902, -0.01286059, -3e-08, 0.04928462, 0.03529288, 0.05285677, 0.03853958, 0.07112604, 0.00516566, -0.00598085, 0.03667394, 0.02136712, 0.01665488, -0.07899886, -0.02417018, 0.10293386, 0.13488491, 0.00140272, 0.05338902, 0.0316888, -0.02112306, -0.04682239, -0.03047531, 0.11084254, 0.03284909, -0.02082311, -0.01321021, 0.03375754, -0.04368555, -0.036698, 0.09119874, -0.03039101, 0.07951652, 0.03983482, 0.02595712, 0.00361478, 0.05008143, 0.07831612, 0.02761605, 0.00927197, -0.06401724, -0.05180075, -0.05129496, 0.03102146, 0.03983335, 0.00109123, -0.05928874, 0.00163707, -0.0931377, -0.09673574, -0.14957963, 0.05342098, -0.04998968, 0.02917849, -0.00938563, 0.00163765, 0.05705127, 0.12669984, 0.00574712, 0.02448412, -0.10131153, -0.03043104, 0.01882762, 0.04563716, -0.03664334, 0.03164556, 0.00935791], [0.00868479, -0.0324934, 0.00565535, 0.0197756, 0.05774445, 0.03078582, -0.05720086, -0.0479194, -0.04664707, 0.00261327, 0.01868065, 0.01802289, -0.00545533, -0.01685091, -0.02795398, 0.01180782, -0.00032527, 0.01579162, 0.01175946, -0.01011285, -0.01258714, -0.06152342, -0.07100994, 0.02792097, 0.1617365, -0.04836809, -0.02198164, 0.00161094, 0.00966983, 0.01922676, -0.00480897, 0.08568072, 0.14904803, 0.0213483, -0.0031439, -0.0457252, 0.06102513, 0.01535256, 0.02258217, -0.06177523, -0.00648939, -0.0056777, 0.00718449, -0.03010145, -0.04221268, 0.10117507, 0.02842759, 0.01003488, -0.06050865, 0.04054123, 0.01341026, 0.05407416, 0.01088241, 0.04637878, 0.01453619, 0.04624895, -0.03763303, -0.06049898, -0.02811952, 0.02793356, 0.00169333, 0.01055554, 0.00991526, 0.03237963, 0.05990658, -0.04719593, 0.00692204, 0.03761563, -0.12038797, 0.07003168, 0.00444169, -0.00389834, -0.0156237, -0.00470687, 0.04003881, -0.07051881, 0.02351904, -0.01488478, 0.03460797, -0.05343084, -0.07283289, 0.07628879, -0.0330104, -0.00401191, -0.04617872, -0.01257952, 0.0837605, -0.07868692, 0.01193344, -0.09458097, -0.01710591, -0.00804476, -0.00241529, -0.0007723, -0.01636472, -0.03473957, -0.01736681, -0.01247413, 0.0406687, 0.061419, -0.01276656, -0.03485221, 0.03254783, -0.03482216, -0.01551596, -0.04379338, 0.03034104, 0.01798384, -0.03099933, 0.04344777, 0.01964181, 0.01137738, -0.00495036, 0.07875516, 0.00735329, -0.08564284, 0.02844783, 0.0276077, 0.07554451, 0.0095714, 0.08640651, 0.04672493, -0.05745062, 0.02486505, 0.0080192, -0.02884183, 0.06487112, -0.0, 0.02787513, -0.06717099, 0.01578104, 0.04911476, 0.0330016, -0.02835262, -0.05074574, 0.00482894, 0.01367533, 0.04957946, 0.03551717, 0.06187773, 0.04217063, -0.01874563, 0.04431302, 0.00747256, -0.10293715, 0.05560799, 0.03603254, 0.00469704, -0.07228822, 0.00378868, -0.04442173, 0.05234215, 0.00875935, -0.07625677, -0.07499401, -0.00206923, 0.01482569, -0.00740325, 0.05854373, 0.03492684, -0.00837476, -0.07752123, -0.11738475, 0.00683131, 0.02536783, -0.00826037, -0.08701396, -0.03348654, -0.06993646, -0.02715574, 0.04779318, -0.11162747, -0.05856573, -0.05163417, 0.03562033, 0.0707233, -0.04496129, 0.10180821, -0.03562927, -0.01472521, 0.02077751, -0.0796297, -0.02043124, 0.03054727, -0.02139458, -0.02261616, 0.01728477, -0.02960665, -0.10826425, -0.11249904, 0.03355037, 0.07899643, 0.0213126, 0.03484984, 0.07615093, 0.03299316, 0.05824649, 0.07903219, -0.0635368, -0.01887708, -0.09726804, 0.03756688, -0.01229597, 0.06708504, -0.00998663, 0.05256357, -0.05563219, 0.00936281, -0.0321042, 0.01402032, 0.07361264, -0.11292859, 0.04473257, 0.04053098, 0.03149609, 0.05596228, 0.08607885, -0.01506729, -0.00710916, -0.07795439, 0.06035227, 0.04854157, 0.02578181, -0.0, -0.10566799, 0.05447165, -0.06026963, 0.03132038, 0.04189738, 0.00411428, 0.12112048, -0.09919745, -0.10030774, 0.06647717, -0.01908009, -0.00318194, 0.06416661, 0.04315198, -0.02833268, -0.03383987, 0.01330704, -0.0029946, 0.06274584, -0.04168938, -0.03615233, 0.02389516, 0.11534197, -0.00577426, 0.05996715, 0.00413777, -0.10960323, 0.06518023, -0.09934463, -0.01014042, -0.02542874, 0.01139176, 0.02270858, -0.03150402, 0.02747173, -0.09943104, -0.14456394, 0.03244253, -0.0088206, -0.0036767, 0.04979508, 0.00617033, -0.01143881, 0.03778255, 0.00616909, -0.0181592, 0.03120732, -0.04051133, 0.05463635, 0.02097823, 0.07251798, 0.04597039, 0.0216772, -0.03650942, -0.03027574, 0.04822052, 0.06650057, -0.01618641, -0.05030594, 0.03834255, -0.09105501, -0.00271562, -0.05024235, 0.14523762, 0.03189262, -0.01422745, -0.01755375, -0.05798142, 0.0706672, 0.00630354, 8.571e-05, -0.03958251, 0.06561951, -0.03088431, -0.01469283, 0.00786952, 0.01329404, 0.02044214, -0.0306132, 0.06355613, 0.00382967, -0.04852493, 0.05825532, 0.01591112, -0.16516981, -0.03656491, -0.05099504, 0.04157123, -0.02937662, 0.09994331, -0.11372072, 0.03339149, -0.03507134, -0.09746621, 0.07382011, -3e-08, 0.11285352, 0.01112439, 0.00405306, -0.0212217, 0.00055818, -0.06157935, 0.01608054, 0.03711906, -0.0036647, 0.15676111, -0.03086026, 0.03451271, 0.02629033, 0.00765462, -0.01889581, 0.04239427, -0.01862784, -0.06784505, -0.06029592, -0.05799663, 0.10029004, 0.04232379, -0.02079549, 0.01267688, -0.04189429, 0.0478942, -0.01459537, 0.06176503, -0.06492206, -0.01188961, 0.05035525, 0.06092864, 0.01330538, -0.03908651, -0.03408031, 0.01932878, -0.03258519, -5.467e-05, 0.0076073, -0.06113718, 0.00680721, 0.05266834, -0.03170758, -0.05403154, -0.04528158, -0.06511845, -0.04133228, -0.00266645, -0.04306322, -0.06264468, 0.00623975, -0.00197147, 0.00443694, -0.0328677, 0.01499982, -0.10492338, 0.01752235, -0.02284306, -0.05317834, -0.02803883, 0.02131348, -0.01296367, 0.03513992, -0.04961844], [-0.00372105, 0.02017334, 0.02980713, -0.0393528, -0.01672171, 0.01554655, -0.06250736, 0.03381364, 0.00952864, 0.04317981, -0.04273401, 0.0789111, 0.07726665, -0.02737936, -0.04383261, 0.06632636, 0.04136932, -0.00172268, -0.11123872, -0.04678196, 0.01841867, -0.06019171, -0.03665329, -0.01978448, 0.01849067, -0.04999884, 0.02477086, -0.01397985, 0.0313044, 0.02090205, -0.02330578, 0.01400881, 0.13627766, -0.02855086, -0.10536369, 0.0129246, -0.01891802, -0.02149025, 0.0496099, -0.0447064, -0.00404506, -0.03910183, -0.00023793, -0.04526543, 0.06899346, 0.10539606, -0.02073373, -0.00913269, -0.00448716, -0.01230685, -0.09522662, 0.033123, 0.03502385, 0.0044177, -0.08611532, 0.01168249, 0.04026045, 0.03691831, -0.01445095, 0.04869579, 0.0653266, -0.03788408, 0.05362406, 0.03819505, 0.11291344, -0.0315336, 0.01227125, 4.495e-05, -0.09025297, 0.04102838, 0.07216016, 0.03669199, -0.02900017, 0.01242105, 0.06917574, -0.0309893, 0.03650737, -0.01735585, 0.05094545, -0.03109989, 0.02082479, -0.00527984, -0.0651086, 0.04986317, -0.01363764, -0.03951995, 0.10233629, -0.09539963, 0.10700821, 0.0113478, 0.01783549, -0.11434687, -0.00266784, 0.03993444, 0.06788897, -0.0054928, -0.03215218, -0.00616196, -0.0730074, 0.04230136, -0.02467631, -0.03661831, -0.01276763, -0.02179809, 0.02900515, -0.04065438, 0.08941405, -0.05856938, 0.03220751, -0.00322396, 0.03760289, -0.00124672, 0.08335226, -0.00920782, -0.01491638, -0.01856699, 0.04399388, 0.08886614, -0.02593, -0.05115311, -0.04345495, -0.09555843, -0.00655929, 0.11517672, 0.04062195, 0.02823781, -0.08157789, -0.0, -0.06087447, -0.09862023, 0.04968793, -0.01649416, 0.0545603, -0.04135228, -0.02307826, -0.06425212, 0.02425137, 0.03777218, -0.06892584, 0.0645136, 0.0456221, 0.01654034, 0.07820009, -0.01186036, -0.07122363, -0.02696018, -0.0256615, -0.02517641, 0.00773525, -0.14925948, -0.04906071, 0.01150149, -0.01331138, -0.0474132, -0.01257974, -0.05049462, 0.01119755, -0.04385263, -0.04768403, 0.04086767, -0.04092047, 0.00662747, 0.02500341, 0.03308269, -0.01254689, 0.03448609, -0.02560646, 0.06530972, -0.03680443, 0.01513223, 0.0628071, -0.07375904, 0.02874496, -0.07289182, 0.06743995, -0.00244428, -0.05712368, -0.03517361, -0.06755415, 0.05200579, 0.01885432, -0.05260814, 0.08514113, -0.02188067, -0.00939268, 0.03675554, -0.01133206, -0.02324553, -0.03884101, -0.06739367, 0.09082469, 0.04379195, 0.05203829, 0.13728651, 0.02582444, 0.03174383, 0.05536693, 0.07652774, -0.02383191, -0.03278005, -0.03999844, 0.00138663, -0.06778581, 0.09571321, -0.01810072, -0.0404437, 0.03455654, -0.01491202, -0.09944537, -0.01899549, 0.03973006, -0.04857076, -0.01489332, -0.03011003, -0.03026725, -0.02490814, 0.06071732, -0.07493649, -0.03871821, 0.01018013, -0.01170008, 0.10832165, -0.05584414, 0.0, -0.10746904, -0.00954206, -0.1324749, -0.020899, 0.01880899, -0.02403856, 0.03596436, -0.04353461, -0.01298913, 0.04792553, 0.00137296, 0.00657199, -0.03178672, 0.0522314, 0.0707549, -0.03586129, 0.03494558, 0.02549305, -0.01137207, -0.04000633, -0.00145188, 0.04886958, 0.0169526, 0.01817008, 0.02453246, 0.02367989, -0.03760876, 0.08380491, -0.05319193, -0.02279134, 0.02545348, -0.02758192, -0.07503336, 0.06964903, 0.04619664, -0.0117255, 0.02768666, -0.04699397, -0.01655557, 0.04118469, 0.02222151, -0.03304743, -0.02242889, 0.0078207, 0.02550348, 0.04101032, 0.00736484, 0.02334646, 0.04752398, -0.03060519, 0.0810798, 0.02733626, -0.00053372, 0.01749431, -0.11112913, -0.05516097, 0.00315834, -0.04555085, -0.00109429, 0.08178107, -0.10641469, -0.00972084, -0.04566237, 0.0546842, -0.12715988, 0.01248551, 0.03148956, 0.04721339, 0.06436629, -0.06147342, 0.08053596, -0.00277658, 0.02263344, 0.00509554, -0.08227648, 0.04829258, 0.06695009, 0.06392755, 0.00145429, 0.00670909, -0.04056754, -0.05880087, 0.06641654, 0.01423349, -0.12322873, 0.0319819, -0.0316025, -0.02561449, -0.00817173, 0.08237221, -0.06095435, 0.0460063, -0.03828843, -0.12184888, -0.06055281, -1e-08, -0.00121999, 0.01526657, 0.08362319, 0.01613015, -0.00296575, -0.00634813, -0.05346908, 0.02580854, -0.0486103, 0.01341777, 0.0048851, -0.00258403, 0.03116325, 0.02568929, 0.03022699, 0.04135514, -0.04604529, -0.03929859, 0.00611797, -0.014399, 0.11734228, -0.04862408, 0.00621286, -0.02576025, 0.02477849, -0.0067633, 0.0058299, 0.01448495, -0.10822123, 0.07345427, 0.01898319, 0.07224297, 0.04454705, -0.01191722, 0.04349712, 0.0152001, 0.02041547, -0.05156709, -0.10445733, -0.01487156, -0.00478972, 0.076177, 0.01383268, -0.0596947, 0.06460033, -0.0097966, 0.06055747, -0.04389119, 0.04334718, -0.04857457, 0.01522929, 0.08319139, 0.06038136, 0.03145161, 0.10287934, -0.08754817, 0.0238775, -0.05713984, -0.02862469, 0.08435009, 0.04563085, -0.000478, 0.00852092, -0.09375551]]

saved_followup_query = 'How does AI affect inequality? How much did those channels change the US Gini coefficient in 2024?'
saved_followup_vector = [0.01279583, -0.05354669, 0.02509943, -0.02741053, -0.04723732, 0.04737959, -0.09947257, 0.04030732, -0.01041623, 0.04886446, -0.07725496, 0.10061719, 0.02764628, 0.00404503, -0.05767688, 0.02185497, 0.0031639, -0.0680584, -0.09564996, -0.01292949, 0.07381207, -0.02104585, -0.01639426, -0.03707679, 0.08556447, -0.0856061, 0.04715952, -0.0350142, 0.03250784, 0.07003628, -0.03597502, 0.06626436, 0.14620104, -0.06749044, -0.08272383, -0.04397725, 0.03174979, -0.04985308, 0.05433045, -0.00635245, 0.03104451, -0.07355725, 0.00093921, -0.03362776, 0.03798972, 0.05456552, 0.02599936, 0.02992648, -0.02031577, 0.03896359, -0.04999634, 0.05662287, 0.0857992, 0.02248107, -0.05888135, -0.00772093, -0.00924858, 0.02781924, 0.03034405, 0.13517126, 0.03625396, -0.03649955, 0.01591424, 0.05619682, 0.06448655, 0.04254338, -0.01365119, 0.04454536, -0.07345646, 0.03178691, 0.05939318, -0.04360556, -0.05422379, -0.03427766, 0.10731184, -0.03166105, 0.06024709, 0.0245446, 0.08925636, -0.05116769, 0.04909398, -0.00633013, -0.05603123, -0.01271279, -0.00822435, 0.02149286, 0.05226491, -0.05441276, 0.08075483, 0.00237581, -0.05685409, -0.06945196, 0.00415715, 0.04582699, 0.05595732, -0.02103871, -0.08213949, -0.02637749, -0.01183419, 0.03565338, -0.02098338, -0.00570879, -0.01387027, -0.05216962, -0.03857104, -0.02705701, 0.04659472, -0.01261948, 0.03931433, 0.0584342, 0.01510995, 0.00305825, 0.02207195, -0.01478722, 0.02955602, -0.01871333, 0.05888715, 0.12489138, 0.04197117, -0.0155922, -0.00461528, -0.05872817, -0.02038269, 0.07789592, 0.00796262, 0.00816204, -0.10635862, -0.0, -0.02747661, -0.04780142, 0.03006057, -0.02591085, -0.00930179, 0.02534231, -0.03381678, -0.05626632, 0.03383647, 0.03435443, -0.04784964, 0.0978389, -0.039984, -0.00991227, 0.11112285, -0.02173232, -0.11554033, 0.00613916, -0.01589069, -0.03151328, 0.01481521, -0.06894672, -0.04180686, -0.00672837, 0.04725911, -0.02600575, -0.00212644, -0.08110083, 0.04045285, -0.00993645, -0.02314144, 0.02255986, -0.02246987, -0.04774209, -0.03360658, 0.01651965, -0.03973375, 0.04570365, -0.05620395, 0.03245489, -0.01707495, 0.0375813, 0.07432435, -0.07958668, 0.02964653, -0.08233707, 0.08951743, 0.00227853, -0.00272283, 0.02984056, -0.09709068, 0.08613072, -0.06530713, -0.05423449, 0.02555042, -0.03435372, -0.04895193, 0.051241, -0.03324562, -0.05087558, -0.07042515, -0.02730504, 0.0434577, 0.00263775, 0.01991785, 0.17208003, -0.00855781, -0.00320451, -0.00812546, 0.07190148, -0.03751953, -0.02855194, -0.09350847, -0.03186999, -0.07307132, 0.07892995, 0.02990631, -0.01410376, 0.06002141, 0.03023863, -0.10118756, -0.02587252, 0.05268399, -0.01091428, 0.07250486, 0.01860275, 0.03142275, 0.00824068, 0.04808208, -0.09969774, -0.00755088, 0.00556391, 0.03069567, 0.10973148, -0.04369637, -0.0, -0.1219576, 0.01025232, -0.04933616, 0.00169385, 0.0102482, -0.02299059, 0.06124127, 0.01222265, -0.00509165, 0.00589574, 0.03204953, -0.01463251, 0.0038177, 0.11605559, 0.0114861, -0.07146069, 0.05183334, -0.02165777, -0.03456992, -0.02261546, 0.06696279, 0.08991975, 0.03946339, 0.01113673, -0.00264591, -0.01288398, 0.00938096, 0.03680215, -0.0307609, -0.04852728, 0.03092852, -0.00521376, -0.12105157, 0.05834119, 0.03778894, 0.01818946, 0.05494082, -0.05166211, -0.08195506, -0.03545776, 0.03204612, -0.00326403, -0.03434947, -0.0198153, -0.00165836, 0.0539337, 0.01058285, 0.05117271, -0.02492334, -0.03275204, 0.05370335, 0.04274716, -0.03206308, 0.03596338, -0.10181331, -0.08079688, 0.10270181, -0.02816759, -0.03380426, 0.04013977, -0.07645494, 0.00796548, -0.04230721, -0.03117048, -0.09508998, -0.03900332, 0.06112903, 0.01832819, 0.02966565, -0.05985509, 0.11267892, -0.04892839, 0.01598057, 0.01638073, -0.0489014, 0.10195103, 0.02622989, 0.03651395, -0.00802963, -0.00660024, -0.05576579, -0.00200382, 0.00235927, -0.01627073, -0.02822835, 0.06520061, 0.02009962, -0.05000022, -0.00335252, 0.08910692, -0.07775764, 0.02980133, -0.08219923, -0.06338149, -0.02839389, -2e-08, 0.00207527, 0.06050803, 0.00805903, 0.01811186, -0.0396858, -0.0339724, -0.08275132, 0.02729924, -0.05001992, -0.00039054, 0.05851255, 0.02148864, 0.05002778, 0.02148902, 0.03295612, -0.00760054, -0.07269927, 0.00638302, 0.00994737, -0.05129047, 0.10814828, -0.01848385, 0.04115957, -0.02371233, -0.03232011, -0.01509062, -0.00931484, 0.0158932, -0.11688941, 0.0205985, 0.02408398, 0.0355942, 0.02331159, -0.05800142, -0.0486818, 0.0223531, -0.01417688, -0.02393234, -0.06041325, -0.00039955, 0.00695942, -0.03419112, -0.0135192, -0.02864947, 0.03656952, -0.04798958, 0.02700435, 0.01331679, 0.08503631, -0.0396033, -0.01618711, 0.07123985, 0.0177147, -0.01737897, 0.1384542, -0.07517149, 0.0028228, -0.07425406, -0.06074677, 0.01361683, 0.0219721, -0.04352958, -0.07725638, -0.04568866]

In [ ]:
texts = [p["text"] for p in passages] + [question]
signature = hashlib.sha256(json.dumps(texts, ensure_ascii=False).encode("utf-8")).hexdigest()
if fresh_models:
    vectors = encode_texts(texts).tolist()
elif signature == saved_input_hash:
    vectors = saved_vectors
else:
    raise ValueError("The example text changed. Enable fresh_models to re-embed it.")

db = sqlite3.connect(":memory:")
db.execute("""CREATE TABLE passages
    (id TEXT PRIMARY KEY, title TEXT, url TEXT, start INTEGER, end INTEGER, text TEXT, vector BLOB)""")
for p, v in zip(passages, vectors[:-1]):
    db.execute("INSERT INTO passages VALUES (?, ?, ?, ?, ?, ?, ?)",
               (p["id"], p["title"], p["url"], p["start"], p["end"], p["text"], struct.pack("<384f", *v)))
db.commit()

## 2. Find passages for the question

A search for *inequality* should be able to find a paper that says *income distribution*. The first search locates passages near the question in the embedding space. A second search, BM25, ranks passages by the words they share with the question, giving rarer words more influence.

The notebook combines the two lists using **reciprocal rank fusion**. Each search contributes a little more weight to passages near the top of its list. In this example the first two passages trade places across the lists and tie after combination. Passage ID settles the tie. Read the three passages passed to the model below. Which one most directly answers the student’s question?

In [ ]:
import re
from collections import Counter

def tokenize(text):
    return re.findall(r"\b\w+\b", text.casefold())

def bm25_scores(query, records):
    counts = {p["id"]: Counter(tokenize(p["text"])) for p in records}
    lengths = {key: sum(counts[key].values()) for key in counts}
    avg_length = sum(lengths.values()) / len(lengths)
    scores = dict.fromkeys(counts, 0.0)
    for term in tokenize(query):
        frequency = sum(term in tokens for tokens in counts.values())
        idf = math.log1p((len(counts) - frequency + 0.5) / (frequency + 0.5))
        for key, tokens in counts.items():
            tf = tokens[term]
            length_factor = 0.25 + 0.75 * lengths[key] / avg_length
            scores[key] += idf * tf * 2.5 / (tf + 1.5 * length_factor)
    return scores

def ranked(scores):
    return sorted(scores, key=lambda key: (-scores[key], key))

def retrieve(query, query_vector):
    dense = {}
    for key, vector in db.execute("SELECT id, vector FROM passages"):
        coords = struct.unpack("<384f", vector)
        dense[key] = sum(x * y for x, y in zip(query_vector, coords))
    sparse = bm25_scores(query, passages)
    # Equal-weight reciprocal rank fusion. Positions start at one.
    combined = Counter()
    for scores in (dense, sparse):
        for rank, key in enumerate(ranked(scores)[:6], start=1):
            combined[key] += 0.5 / (60 + rank)
    return ranked(dense)[:3], ranked(sparse)[:3], ranked(combined)[:3]

dense_ids, sparse_ids, selected = retrieve(question, vectors[-1])
print("Question:", question)
print("Search by meaning:", dense_ids)
print("Search by words (BM25):", sparse_ids)
print("Combined ranking:", selected)
passage_text = {p["id"]: p["text"] for p in passages}
for passage_id in selected:
    print(f"\n[{passage_id}] {passage_text[passage_id]}")

## 3. Write an answer from the passages

The model receives the original question, the three passages printed above and an instruction to explain the mechanisms in two sentences with passage references. The text sent to the model is called a *prompt*. In the default run the notebook reconstructs that prompt, checks that it matches the saved example and displays Qwen’s original draft. With local inference enabled, Qwen writes a new answer from the assembled prompt.

Read the draft alongside `distribution-1`. It identifies two routes to changes in household inequality. Innovators may receive some of the surplus, and changes in wages and returns to capital may affect households differently depending on what they own. The draft also discusses development and growth. Decide which parts you would keep in a two-sentence answer to the student.

In [ ]:
sources = {p["id"]: p for p in passages}
history = []
writer = load_writer() if fresh_models else None

def ask(message, *, follow_up=False):
    if follow_up and not history:
        raise ValueError("A follow-up needs an earlier question.")
    search_text = history[-1]["message"] + " " + message if follow_up else message
    if search_text == question:
        query_vector = vectors[-1]
    elif fresh_models:
        query_vector = encode_texts([search_text])[0]
    elif search_text == saved_followup_query:
        query_vector = saved_followup_vector
    else:
        raise ValueError("Enable fresh_models to ask a new question.")

    _, _, selected_ids = retrieve(search_text, query_vector)
    evidence = "\n".join(f"[{key}] {sources[key]['text']}" for key in selected_ids)
    years = set(re.findall(r"\b(?:19|20)\d{2}\b", message))
    missing = sorted(year for year in years if year not in evidence)
    if missing:
        answer = "The retrieved notes do not cover " + ", ".join(missing) + ". They cannot support the requested estimate."
    else:
        prompt = ("Answer the economics question using only these notes. If the evidence is missing, say so. "
                  "Give two sentences, naming the mechanisms and citing the supporting passage IDs.\n\n"
                  + evidence + "\n\nQuestion: " + search_text)
        if fresh_models:
            answer = writer(prompt)
        elif hashlib.sha256(prompt.encode("utf-8")).hexdigest() == saved_prompt_hash:
            answer = saved_answer
        else:
            raise ValueError("The context changed. Enable fresh_models for a new draft.")
    turn = {"message": message, "search": search_text, "retrieved": selected_ids, "answer": answer}
    history.append(turn)
    return turn

first = ask(question)
print("Model draft:", first["answer"])
for key in first["retrieved"]:
    print(key, sources[key]["url"])

## 4. Ask a follow-up

The student now asks, *How much did those channels change the US Gini coefficient in 2024?* To search for “those channels,” the notebook includes the preceding question about AI and inequality. It retrieves three passages again.

The retrieved notes describe mechanisms and omit the requested year. A rule in this example checks the year before asking the model to write. When it is missing, the code returns the message below. Even if a passage mentioned 2024, a reader would still need to find the right country, measure and estimate to support an answer.

In [ ]:
second = ask('How much did those channels change the US Gini coefficient in 2024?', follow_up=True)
print("Search:", second["search"])
print("Passages reviewed:", second["retrieved"])
print("Reply:", second["answer"])

## From a worked example to a course tutor

A course tutor could use the instructor’s assigned readings and lecture notes as its library. After explaining the two mechanisms, it could ask a student, *How would the prediction change if every household owned equal shares of capital?* A conversational interface could pass the student’s next question and relevant conversation history to the retriever, then show the supporting passages beside the reply. This notebook demonstrates that exchange with two scripted turns.

The same approach could help a researcher draft a literature review using passages on assumptions and findings, or locate a model equation and the corresponding routine in a library that includes replication code.

**Try it offline.** Read the distribution passage and mark the sentences in the first reply that you could defend from that passage alone. Compare the evidence needed for a mechanism with the evidence needed to estimate the Gini coefficient.

**Try a new question with local models.** After enabling `fresh_models` in step 1, run all cells and enter `ask("How can automation affect labor’s share of income?")` in a new cell. Inspect the returned passage IDs along with the answer. Use `follow_up=True` for a subsequent question that refers to the previous one.